PCA clean and transfer function on the MeerKLASS L2021 cube, cropped to the
drift-scan footprint. This is the benchmark the Gibbs sampler is compared
against: `T_PCA` is the fraction of an injected mock that survives a PCA clean,
and `2_gibbs_sampling.ipynb` measures the equivalent `T_Gibbs` for the sampler.

The k-binning here is the box-based `make_kbins`, not the footprint-aware
`kbins_from_crop` the sampler uses. That is deliberate -- the published PCA
transfer function is calibrated against these bins -- but it does mean the two
notebooks' k axes are not identical. See docs/STATUS.md.


In [ ]:
# Run from the repo without installing: `pip install -e ..` makes this unnecessary.
import sys, pathlib
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import numpy.fft as fft
import matplotlib.pyplot as plt
from tqdm import tqdm

import fastbox
from fastbox.box import CosmoBox, default_cosmo

from imgibbs import (
    signal_covariance_sampler as SCS,
    make_kbins, power_spectrum, survey_grid, data_path, load_l2021_cube,
)

np.random.seed(41)


## Loading the data cube 

In [ ]:
# ====================== SET THE GRID HERE ==============================
# 72 channels here, against 250 in the Gibbs notebook: this benchmark is a
# self-contained PCA + transfer-function measurement and does not consume
# S_starting_point, so it is free to use a shorter band. Widen it to
# slice(0, 250) to compare like for like with the sampler.
CROP = (slice(33, 103), slice(14, 59), slice(0, 72))
# =======================================================================

full_cube = load_l2021_cube()
data_cube = full_cube[CROP]

shape = data_cube.shape
valid = data_cube != 0
n_full, n_crop = int((full_cube != 0).sum()), int(valid.sum())

# One mask reused for every P(k) below, so the uncleaned and cleaned spectra
# are built on exactly the same footprint.
mask = valid

print(f'full cube     : {full_cube.shape}   fill {(full_cube != 0).mean()*100:.2f}%')
print(f'cropped       : {shape}   fill {valid.mean()*100:.2f}%')
print(f'valid voxels  : {n_full:,} -> {n_crop:,}  (lost {n_full - n_crop:,})')
print(f'grid voxels   : {full_cube.size:,} -> {data_cube.size:,}')


In [ ]:
# Same shared derivation the other two notebooks use.
grid = survey_grid(CROP, shape)

box_dims = grid.box_dims
z_lo, z_hi, z_mid = grid.z_lo, grid.z_hi, grid.z_mid
freqs, chans = grid.freqs, grid.chans

print(grid.summary())


## PCA

In [ ]:
def PCA_clean(data_cube, no_modes):
    shape = data_cube.shape
    nf    = shape[2]                              # was hardcoded to 72

    # PCA eigenvectors from frequency-frequency covariance (matches reference notebook)
    full_valid = np.all(data_cube != 0, axis=2)   # pixels non-zero across all freq channels
    d_valid    = data_cube[full_valid]            # (N_valid, nf)

    # NOTE: no mean spectrum is subtracted here, unlike also_PCA_clean() and
    # fastbox.filters.pca_filter(), which both subtract it before the
    # eigendecomposition and add it back into the foreground model.
    C = (d_valid.T @ d_valid) / d_valid.shape[0]          # (nf, nf)
    eigenvalues_all, eigenvectors_all = np.linalg.eigh(C)
    idx_sorted       = np.argsort(eigenvalues_all)[::-1]
    eigenvalues_all  = eigenvalues_all[idx_sorted]
    eigenvectors_all = eigenvectors_all[:, idx_sorted]

    evecs = eigenvectors_all[:, :no_modes].T   # (no_modes, nf) - orthonormal rows

    # reshape on nf, not 72. With a 500-channel cube the old line silently
    # reshaped to (-1, 72) and then died on the matmul below.
    d_2d    = data_cube.reshape(-1, nf)
    fg_amps = d_2d @ evecs.T                   # (Npix, no_modes)
    fit     = fg_amps @ evecs                  # (Npix, nf)

    # transforming back to a cube
    fg_cube = fit.reshape(shape)

    cleaned_cube = data_cube - fg_cube

    return cleaned_cube, evecs


In [ ]:
def also_PCA_clean(data_cube, no_modes):

    #Calculating the Covariance Matrix for the Mean substracted data. Code adapted from Fastbox
    field = data_cube

    d = np.reshape(field, (field.shape[0]*field.shape[1],field.shape[2])).T #(N_freq, N_x_Pix*N_y_Pix)

    # Calculate average spectrum (avg. over pixels, as a function of frequency)
    d_mean = np.mean(d, axis=-1)[:,np.newaxis]

    # Calculate freq-freq covariance matrix
    x = d - d_mean
    cov = np.cov(x) # (Nfreqs x Nfreqs)

    #Perform Eigendecomposition of the Covariance matrix
    eigval, eigvec = np.linalg.eig(cov)

    #sorting the eigenvalues and eigenvectors in descending order:
    idx = np.argsort(eigval)[::-1]
    eigval = eigval[idx]
    eigvec = eigvec[:,idx]

    #Creating the Mixing Matrix
    A = eigvec[:,:no_modes] #(N_freq,N_modes)

    #Performing the cleaning 🧼
    S = np.dot(A.T,x)
    x_fg = np.dot(A,S) + d_mean
    fg_cube= np.reshape(x_fg.T, (field.shape))

    #print('fg_cube:', fg_cube.shape)
    cleaned_cube =  field - fg_cube

    #print('clean_cube:', cleaned_cube.shape)
    #cleaned_cube = np.reshape(cleaned_cube, (field.shape[2],field.shape[1],field.shape[0]) ).T

    return cleaned_cube, A

In [ ]:
N_PCA_MODES = 8   # named so the clean and the transfer function agree

cleaned_cube, eigvec = PCA_clean(data_cube, N_PCA_MODES)
also_cleaned_cube, also_eigvec = also_PCA_clean(data_cube, N_PCA_MODES)


In [ ]:
print(eigvec.shape)

In [ ]:
no_modes_cmp = 8
fig, ax = plt.subplots(no_modes_cmp, 1, figsize=(20, 30), dpi=300)

# build the frequency axis from the cube actually loaded, rather than
# assuming 72 channels starting at 550.
new_freq = freqs

for i in range(no_modes_cmp):
    g = eigvec[i, :]            # GPCA (Geoff)  -> (no_modes, nf)
    c = also_eigvec[:, i]       # CPCA (Caelin) -> (nf, no_modes)
    # eigenvectors are only defined up to a sign, so flip CPCA onto GPCA
    # before overplotting - otherwise half the panels look like disagreements.
    if np.dot(g, c) < 0:
        c = -c
    ax[i].plot(new_freq, g, lw=3, color='red',  label='GPCA')
    ax[i].plot(new_freq, c,       color='blue', label='CPCA')
    ax[i].set_ylabel(f'mode {i}')
    ax[i].legend()

ax[-1].set_xlabel('frequency [MHz]')
plt.tight_layout()


## Plotting Eigenvectors

In [ ]:
no_modes = 4
fig, ax = plt.subplots(no_modes, 1, figsize=(20, 16), dpi=300)

new_freq = freqs   # derived from the cube, was freq_channel[550:550+72]

colours = ['indigo','blueviolet','mediumpurple','darkslateblue','darkorchid','slateblue']

# loop over no_modes (was range(6) against no_modes axes), and index
# eigvec as (no_modes, nf) - it was being sliced as if it were (nf, no_modes).
for i in range(no_modes):
    ax[i].plot(new_freq, eigvec[i, :], color=colours[i], label=rf'$\vec{{x}}_{i+1}$')
    ax[i].legend()

ax[-1].set_xlabel('frequency [MHz]')
plt.tight_layout()


## FastBox Eigenvectors

In [ ]:
fb_cleaned_cube, fb_evecs, _ = fastbox.filters.pca_filter(data_cube, 
                                                             nmodes=no_modes, 
                                                             return_filter=True)

In [ ]:
fig, ax = plt.subplots(no_modes, 1, figsize=(20, 16), dpi=300)

new_freq = freqs   # derived from the cube

colours = ['midnightblue','blue','mediumslateblue','darkslateblue','darkblue','slateblue']

# range(no_modes), was range(6) against no_modes axes.
# fb_evecs has shape (nf, no_modes), so the column indexing here is correct.
for i in range(no_modes):
    ax[i].plot(new_freq, fb_evecs[:, i], color=colours[i], label=rf'$\vec{{x}}_{i+1}$')
    ax[i].legend()

ax[-1].set_xlabel('frequency [MHz]')
plt.tight_layout()


## Difference between my PCA and Fastbox

In [ ]:
fig, ax = plt.subplots(no_modes, 1, figsize=(20, 16), dpi=300)

new_freq = freqs   # derived from the cube

colours = ['midnightblue','blue','mediumslateblue','darkslateblue','darkblue','slateblue']

# eigvec is (no_modes, nf) and fb_evecs is (nf, no_modes), so transpose
# before subtracting. The old line broadcast (nf, no_modes) against
# (no_modes, no_modes) and raised.
# sign-align first - eigenvectors are only defined up to a sign.
mine = eigvec[:no_modes, :].T.copy()                  # (nf, no_modes)
for i in range(no_modes):
    if np.dot(mine[:, i], fb_evecs[:, i]) < 0:
        mine[:, i] *= -1

residuals = fb_evecs[:, :no_modes] - mine

# NOTE: these will not go to zero. fastbox.filters.pca_filter subtracts the
# mean spectrum before the eigendecomposition; PCA_clean above does not.
for i in range(no_modes):
    ax[i].plot(new_freq, residuals[:, i], color=colours[i], label=rf'residual $\vec{{x}}_{i+1}$')
    ax[i].legend()

ax[-1].set_xlabel('frequency [MHz]')
plt.tight_layout()


## Comparing Cleaned Cube with Original Cube 

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20,6), dpi=300)

chan = shape[2] // 2   # was hardcoded 35

# slices
data_slice       = data_cube[:, :, chan]
cleaned_slice    = cleaned_cube[:, :, chan]
fb_cleaned_slice = fb_cleaned_cube[:, :, chan]

# mask zeros for plotting
data_slice_masked    = np.ma.masked_where(data_slice == 0, data_slice)
cleaned_slice_masked = np.ma.masked_where(data_slice == 0, cleaned_slice)
# was masking fb_cleaned_slice but then plotting cleaned_slice, so this
# panel was a duplicate of the middle one.
fb_slice_masked      = np.ma.masked_where(data_slice == 0, fb_cleaned_slice)

# optional: make masked regions white
cmap = plt.cm.inferno.copy()
cmap.set_bad(color='white')

# plots
data_plot = ax[0].matshow(data_slice_masked.T, cmap=cmap)
fig.colorbar(data_plot, ax=ax[0], label='T[K]')
# .T on the cleaned panels too, so all three share an orientation.
pca_cleaned_plot = ax[1].matshow(cleaned_slice_masked.T*1e3, cmap=cmap)
fig.colorbar(pca_cleaned_plot, ax=ax[1], label='T[mK]')
fb_cleaned_plot = ax[2].matshow(fb_slice_masked.T*1e3, cmap=cmap)
# colorbar was built from pca_cleaned_plot, so it showed the wrong scale.
fig.colorbar(fb_cleaned_plot, ax=ax[2], label='T[mK]')

# titles
ax[0].set_title('Calibrated Data Slice')
ax[1].set_title('PCA Cleaned Data Slice')
ax[2].set_title('Fastbox PCA Cleaned Data Slice')

plt.tight_layout()
plt.show()


In [ ]:
#Residuals between my pca and fastbox

## Comparing the Power Spectrum between the Uncleaned, Cleaned Data Slice and Fastbox HI Cube

In [ ]:
#Since the MeerKAT data is in K the fastbox cube is converted to K from mK

box = CosmoBox(cosmo=default_cosmo, box_scale=box_dims, nsamp=shape,
               redshift=z_mid, realise_now=False)
print(f'box.N     = {box.N}      (tuple -> non-cubic grid)')
print(f'box.shape = {box.shape}')

# (a) Gaussian density field
box.realise_density()

# (b) scale by the H I bias
tracer = fastbox.tracers.HITracer(box)
delta_hi = box.delta_x * tracer.bias_HI()

# (c) log-normal transform
delta_ln = box.lognormal(delta_hi)

# (d) radial velocity field (from the Gaussian density field)
vel_k = box.realise_velocity(delta_x=box.delta_x, inplace=True)
vel_z = fft.ifftn(vel_k[2]).real

# (e) into redshift space
delta_s = box.redshift_space_density(delta_x=delta_ln.real, velocity_z=vel_z,
                                     sigma_nl=120., method='linear')

# (f) brightness temperature, mK -> K to match the data cube
Tb_mK = tracer.signal_amplitude()
signal_cube = (Tb_mK * (1. + delta_s)) / 1000.0

print(f'\nTb(z={z_mid:.3f})   = {Tb_mK:.4f} mK')
print(f'signal_cube     : {signal_cube.shape}')
print(f'  mean {signal_cube.mean():.3e} K   std {signal_cube.std():.3e} K')
print(f'  data cube mean {data_cube[data_cube!=0].mean():.3f} K '
      f'(foreground dominated, ~10^4 x larger)')

In [ ]:
n_k_bins = 14   #12       # or None to auto-pick the largest value that works


def _min_modes(nb):
    """Modes in the sparsest bin at this bin count."""
    _, ix = make_kbins(shape, nb, box_dims=box_dims)
    return min((ix == b).sum() for b in np.unique(ix)[1:])


if n_k_bins is None:
    workable = [nb for nb in range(4, 31) if _min_modes(nb) > 2]
    n_k_bins = max(workable) if workable else 4
    print(f'auto-selected n_k_bins = {n_k_bins}')

sig_k2, idxs2 = make_kbins(shape, n_k_bins, box_dims=box_dims)

counts = np.array([(idxs2 == b).sum() for b in np.unique(idxs2)[1:]])
print(f'n_k_bins = {n_k_bins}')
print(f'modes per bin : {counts}')
print(f'lowest bin    : {counts.min()} modes '
      f'({"OK" if counts.min() > 2 else "TOO FEW - SCS will assert"})')

In [ ]:
#FASTBOX SIMULATED HI POWER SPECTRUM
# shared estimator, so all three spectra are formed identically.
# the simulation is unmasked, so mask=None uses every voxel.
Pk_true, Pk_err_true, n_modes_true = power_spectrum(
    signal_cube, sig_k2, idxs2, box_dims, mask=None)

# NOTE: the sim fills the full box while the data only covers ~59% of it. If
# you want a like-for-like comparison rather than a reference curve, pass
# mask=mask here too and accept the same footprint-induced mode coupling.


In [ ]:
#PCA CLEANED CUBE POWER SPECTRUM
# identical estimator and identical mask to the uncleaned cube below.
Pk_clean, Pk_err_clean, n_modes_clean = power_spectrum(
    cleaned_cube, sig_k2, idxs2, box_dims, mask=mask)


In [ ]:
#UNCLEANED DATA CUBE POWER SPECTRUM
# identical estimator and identical mask to the cleaned cube above.
Pk_unclean, Pk_err_unclean, n_modes_unclean = power_spectrum(
    data_cube, sig_k2, idxs2, box_dims, mask=mask)

# the two spectra must be built on the same modes - assert it rather
# than trusting it, since a stale cube of the wrong shape is exactly
# what went wrong here before.
assert cleaned_cube.shape == data_cube.shape == tuple(shape)
assert np.array_equal(n_modes_clean, n_modes_unclean)
print('modes per bin :', n_modes_unclean)
print('clean / unclean ratio:')
for k, r in zip(sig_k2, Pk_clean / Pk_unclean):
    print(f'  k = {k:7.4f}   {r:.3e}')


In [ ]:
print('Pk_err_true   :', Pk_err_true)
print('Pk_err_clean  :', Pk_err_clean)
print('Pk_err_unclean:', Pk_err_unclean)


In [ ]:
print(sig_k2)

In [ ]:
# nbins=14

# Calculate the 1-dimensional 21cm power spectrum of the fastbox cube 
# sig_k, sig_pk, sig_stddev, idxs = box.binned_power_spectrum(delta_x=signal_cube, nbins=nbins)


# Find the power spectrum of the cleaned data cube
# mK_cleaned_cube=cleaned_cube*1e3
# pca_k, pca_pk, proc_stddev, idxs = box.binned_power_spectrum(delta_x=mK_cleaned_cube, nbins=nbins)

# Find the power spectrum of the uncleaned data cube
# mK_data_cube=data_cube*1e3
# no_pca_k, no_pca_pk, no_proc_stddev, idxs = box.binned_power_spectrum(delta_x=mK_data_cube, nbins=nbins)

In [ ]:
# ---------------------------------------------------------------------
# Why the cleaned P(k) collapses at low k with a 72-channel band.
#
# The radial extent is only Lz ~ 74 Mpc, so the smallest non-zero radial
# wavenumber is 2*pi/Lz ~ 0.085 1/Mpc. Every mode below that has k_par
# EXACTLY zero, i.e. it is the frequency-average of the map. PCA removes
# the dominant smooth frequency eigenmodes, which annihilates the
# frequency-average almost exactly - so those bins go to ~1e-15 of the
# uncleaned power. That is total, irrecoverable signal loss, not a bug,
# and no transfer function can undo it.
#
# Below k_par_min the cleaned points carry no information. Widen the
# frequency crop if you need those scales.
# ---------------------------------------------------------------------
k_par_min = 2.*np.pi / box_dims[2]

KX, KY, KZ = np.meshgrid(*[2*np.pi*np.fft.fftfreq(n)*n/L
                           for n, L in zip(shape, box_dims)], indexing='ij')
kz_flat = KZ.flatten()

print(f'2*pi/Lz = {k_par_min:.4f} 1/Mpc   (Lz = {box_dims[2]:.1f} Mpc)\n')
print(' bin      k     Nmodes   k_par==0    P_clean/P_unclean')
for b in range(1, n_k_bins + 1):
    sel = (idxs2 == b)
    if not sel.sum():
        print(f'{b:4d} {sig_k2[b-1]:8.4f} {0:8d}        -              -')
        continue
    frac = np.mean(kz_flat[sel] == 0) * 100
    print(f'{b:4d} {sig_k2[b-1]:8.4f} {sel.sum():8d} {frac:8.1f}%   '
          f'{Pk_clean[b-1]/Pk_unclean[b-1]:.3e}')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)

# Plot power spectra

# bins with only a couple of modes are pure noise - drop them from the
# plot rather than letting them anchor the eye at low k.
MIN_MODES = 8
good = n_modes_unclean >= MIN_MODES

#Fastbox
ax.errorbar(sig_k2[good], Pk_true[good], yerr=Pk_err_true[good],
            color='gold', marker='.', ls='-', label="FastBox P(k)")

#Uncleaned
ax.errorbar(sig_k2[good], Pk_unclean[good], yerr=Pk_err_unclean[good],
            color='sienna', marker='.', label="Uncleaned data P(k)")

#Cleaned
ax.errorbar(sig_k2[good], Pk_clean[good], yerr=Pk_err_clean[good],
            color='darkgoldenrod', marker='.', label="PCA-cleaned data P(k)")

# mark where k_par becomes resolvable - to its left the cleaned points
# are pure k_par=0 modes that PCA has removed entirely.
ax.axvline(k_par_min, color='grey', ls=':', lw=1)
ax.text(k_par_min, ax.get_ylim()[1], r'  $2\pi/L_z$', color='grey',
        va='top', ha='left', fontsize=9)

ax.legend(fontsize=10)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("k [Mpc$^{-1}$]", size=16)
# the estimator now carries the volume factor, so this really is P(k).
# the cubes are in K, hence K^2 Mpc^3.
ax.set_ylabel("P(k) [K$^2$ Mpc$^3$]", size=16)

print(f'plotted {good.sum()}/{len(good)} bins (dropped bins with < {MIN_MODES} modes)')


## Applying the TF 

In [ ]:
#Define a function which generates a mock/simulated 21cm field
# was hardwired to a 72^3 box at z=0.39 with a 72/256 box_scale fudge,
# which does not match the cropped data grid. Use the same box geometry,
# shape and redshift as the data so the injection below broadcasts.
# returns K (not mK) to match data_cube.
def mock():
    mbox = CosmoBox(cosmo=default_cosmo, box_scale=box_dims, nsamp=shape,
                    redshift=z_mid, realise_now=False)
    mbox.realise_density()

    tracer = fastbox.tracers.HITracer(mbox)
    delta_hi = mbox.delta_x * tracer.bias_HI()

    delta_ln = mbox.lognormal(delta_hi)

    vel_k = mbox.realise_velocity(delta_x=mbox.delta_x, inplace=True)
    vel_z = fft.ifftn(vel_k[2]).real

    delta_s = mbox.redshift_space_density(delta_x=delta_ln.real, velocity_z=vel_z,
                                          sigma_nl=120., method='linear')

    return (tracer.signal_amplitude() * (1. + delta_s)) / 1000.0   # mK -> K


In [ ]:
def TF():
    mock_s = mock()                  # mock 21cm cube, K, same grid as the data

    # both cubes are in K now (the old version referenced mK_data_cube /
    # mK_cleaned_cube, which were only defined in a commented-out cell).
    Inj = data_cube + mock_s
    Inj[~mask] = 0.0                 # keep the injection on the survey footprint

    # clean with the same number of modes as the main clean (was 6 vs 8).
    cleaned_cube_inj, _ = PCA_clean(Inj, N_PCA_MODES)

    X_m_clean = cleaned_cube_inj - cleaned_cube

    # the old version binned the cross-power with fastbox's own k-grid
    # (box.k / box.boxfactor / box.kmin..kmax) while everything else in
    # the notebook used sig_k2/idxs2. Use the shared estimator for both
    # the numerator and the denominator so T(k) is a ratio of like things.
    P_cross, _, _ = power_spectrum(X_m_clean, sig_k2, idxs2, box_dims,
                                   mask=mask, cube2=mock_s)
    P_mock,  _, _ = power_spectrum(mock_s, sig_k2, idxs2, box_dims, mask=mask)

    return P_cross / P_mock


In [ ]:
# the transfer function now shares the notebook's k-bins, so there is
# no separate `nbins` to keep in sync.
nbins = n_k_bins


In [ ]:
N_mocks = 100                          # how many 21cm realisations to use
T_s = np.zeros((N_mocks, n_k_bins))    # n_k_bins, was nbins-1

# Generate the transfer functions
for uu in tqdm(range(0, N_mocks)):
    T_s[uu] = TF()

T_m = np.mean(T_s, axis=0)             # Mean transfer function

# Pk_clean / Pk_err_clean, was pca_pk / proc_stddev from the
# commented-out fastbox cell.
corrected_PS  = Pk_clean / T_m                    # TF-corrected power spectrum
corrected_err = np.std(Pk_clean / T_s, axis=0)    # scatter over realisations


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=160)

# Plot power spectra
# sig_k2 / Pk_clean / Pk_err_clean throughout, and reuse the same
# `good` mask as the plot above so both figures show the same bins.

# below 2*pi/Lz the transfer function is ~0 (PCA removed those modes
# entirely), so Pk_clean/T_m is numerical noise and comes out negative.
# Drop non-positive points rather than letting the log axis silently
# swallow them.
tf_good = good & (corrected_PS > 0)

#PCA + TF cleaned
ax.errorbar(sig_k2[tf_good], corrected_PS[tf_good], yerr=corrected_err[tf_good],
            color='mediumblue', marker='*', label="TF Corrected data P(k)")

#PCA Cleaned
ax.errorbar(sig_k2[good], Pk_clean[good], yerr=Pk_err_clean[good],
            color='mediumturquoise', marker='x', label="PCA-cleaned data P(k)")

#Fastbox
ax.errorbar(sig_k2[good], Pk_true[good], yerr=Pk_err_true[good],
            color='forestgreen', marker='.', ls='-', label="FastBox P(k)")

ax.legend(fontsize=10)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("k [Mpc$^{-1}$]", size=16)
ax.axvline(k_par_min, color='grey', ls=':', lw=1)
ax.set_ylabel("P(k) [K$^2$ Mpc$^3$]", size=16)

print('transfer function T(k):')
for k, t in zip(sig_k2, T_m):
    print(f'  k = {k:7.4f}   T = {t: .4f}')


In [ ]:
np.save(data_path('TF_Pk_L2021.npy'), corrected_PS)

In [ ]:
#np.save(data_path('Fastbox_Pk.npy'),sig_pk)

In [ ]:
np.save(data_path('TF_k_L2021.npy'), sig_k2)   # sig_k2, was undefined sig_k


In [ ]:
np.save(data_path('TF_err_L2021.npy'), corrected_err)